In [ ]:
# Read Counts

In [95]:
library(dplyr)
library(stringr)
library(tidyr)

In [ ]:
# raw and trimmed reads were written via script - found in 0QC_trimming-all
    # combining and pivoting to make easier to read

In [72]:
setwd('/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/')
reads<-read.csv('reads.csv')
sampleids<-read.table('sampleids.txt', sep = "",header = FALSE)
sampleids$sampleid <- sampleids$V1
sampleids$V1 <- NULL

In [75]:
# make sure raw read counts R1 and R2 match
raw_read_match<-reads %>%
    filter(step == 'raw') %>%
    # add base sampleid to match by
    mutate(sampleid = str_remove(sample, "_R[12]_001.fastq")) %>%
    group_by(sampleid) %>%
    # make sure there are 2 seqs per sameplid and that there is only 1 distinct value (meaning they have the same read count)
    mutate(reads_match = case_when(
      n() == 2 & n_distinct(read_count) == 1 ~ TRUE,
      TRUE ~ FALSE
  )) %>%
  ungroup()

# make sure there are no falses
raw_read_match %>%
    filter(reads_match == FALSE)

# perfect they all match 

sample,read_count,step,sampleid,reads_match
<chr>,<int>,<chr>,<chr>,<lgl>


In [98]:
# check sample numbers match 
nsample<-nrow(sampleids) # 225 samples
nsample*2

nrow(reads%>%filter(step=='raw'))
# extra sample is "Undetermined"

missing_id <- setdiff(raw_sample, sampleids$sample)
missing_id

[1] 450

[1] 452

[1] "Undetermined"

In [102]:
# filter for just one Read per sample
raw_reads <-raw_read_match %>%
    group_by(sampleid, read_count) %>%
    slice_head(n=1) %>%
    ungroup() %>%
    select(-sample,-reads_match)
nrow(raw_reads)

In [111]:
# separate trimmed step 
trimmed<-reads %>%
    filter(step == 'trimmed') %>%
    # add base ID 
    mutate(sampleid = str_remove(sample, "_R[1]_001")) %>%
    select(-sample)
# and add raw back in 
reads_all<-rbind(raw_reads,trimmed)

In [120]:
# then pivot
reads_wide<-reads_all%>%
    pivot_wider(names_from = step,
                values_from = read_count)
print(reads_wide)

# A tibble: 226 × 3
   sampleid                        raw   trimmed
   <chr>                         <int>     <int>
 1 052022_BEL_CBC_T1_10_PSTR  68572842  68449647
 2 052022_BEL_CBC_T1_11_PSTR  48741905  48660782
 3 052022_BEL_CBC_T1_12_MCAV 166267321 165624856
 4 052022_BEL_CBC_T1_13_MCAV  86263796  86076666
 5 052022_BEL_CBC_T1_1_PAST   85047990  84808864
 6 052022_BEL_CBC_T1_34_PAST  31359387  31274591
 7 052022_BEL_CBC_T1_35_OANN  55101910  55033169
 8 052022_BEL_CBC_T1_39_MCAV  83479374  83235507
 9 052022_BEL_CBC_T1_40_MCAV  68897750  68586963
10 052022_BEL_CBC_T1_41_OANN  40011606  39927902
# ℹ 216 more rows


In [ ]:
# rewrite long and wide format

In [124]:
write.table(reads_all,'reads_filtered.txt', sep = "\t",row.names = FALSE, quote = FALSE)
write.table(reads_wide,'reads_wide.txt', sep = "\t",row.names = FALSE, quote = FALSE)

In [132]:
# percentage lost in trimming step?
reads_wide<-reads_wide %>%
    mutate(pct_yield = round((trimmed / raw) * 100,2))
reads_wide %>% arrange(pct_yield)

sampleid,raw,trimmed,pct_yield
<chr>,<int>,<int>,<dbl>
7_3_Neg,6030707,5925379,98.25
7_11_Neg,21262507,21046368,98.98
052022_BEL_CBC_T2_60_PAST,104718356,103807571,99.13
052022_BEL_CBC_T1_52_PAST,94884365,94301477,99.39
052022_BEL_CBC_T1_61_PAST,74286118,73869279,99.44
062019_BEL_CBC_T2_2_PAST,28081060,27923531,99.44
122022_BEL_CBC_T1_150_PAST,33672160,33493920,99.47
052022_BEL_CBC_T2_10_MCAV,86868928,86447447,99.51
052022_BEL_CBC_T3_40_MCAV,87769648,87360332,99.53
